# Session 6 — Individual exercise: charging-zone availability

**Business task.** An electric-car rental company consults an external ChargeWatch dashboard to decide whether to recommend the **North or South zone**, based on the number of connectors available at the page's reference time. This comparison could be repeated each morning. The teaching page is one synthetic **historical snapshot**, not live availability: make a recommendation for its stated reference time only.

Work **individually**. All seven students have the same case. AI and documentation are allowed, but your submission must show which choices you verified yourself. You have approximately 40 minutes. Adapt the Playwright sequence demonstrated by the teacher.


## Checkpoint 0 — Start the controlled page

Run the setup cell; it starts a local server. Open the printed URL in a browser. The required packages and Chromium should already have been installed before class.


In [ ]:
from pathlib import Path
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
import sys

def find_sources():
    # Jupyter may start in the package root or inside the notebooks folder.
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "S6_sources"):
            if (candidate / "data" / "student" / "index.html").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Open the extracted S6 package folder in VS Code, then run this cell again.")

SOURCE_ROOT = find_sources()

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass  # Keep repeated browser requests out of the projected notebook.

if "s6_server" not in globals():
    # Port 0 lets the OS choose an available port on this computer.
    handler = partial(QuietHandler, directory=str(SOURCE_ROOT))
    s6_server = ThreadingHTTPServer(("127.0.0.1", 0), handler)
    Thread(target=s6_server.serve_forever, daemon=True).start()

BASE_URL = f"http://127.0.0.1:{s6_server.server_port}"
print("Local page:", BASE_URL)
print("Notebook Python:", sys.executable)

STUDENT_URL = BASE_URL + "/data/student/"
print("Student source:", STUDENT_URL)


## Checkpoint 1 — Define the observation and diagnose the source

Your final comparison needs **one row per zone**. To build and check it, collect **one row per real site** first. Choose one site identifier visible in the browser, then search for it in the first HTML response. Record what you observed.


In [ ]:
import requests

response = requests.get(STUDENT_URL, timeout=10)
response.raise_for_status()
raw_html = response.text
print("HTTP status:", response.status_code)
print("First 500 HTML characters:\n", raw_html[:500])
target_site_id = ""  # TODO: copy one real identifier visible in the browser.
if target_site_id:
    print("Identifier in initial HTML:", target_site_id in raw_html)


**Your evidence:** State the identifier searched, where it appears in the browser, and whether it appears in the initial HTML. Do not treat HTTP 200 as proof that the target records were returned.

## Checkpoint 2 — Choose a collection channel

An alternative [weekly CSV summary](../data/student/weekly_summary.csv) is linked on the page. Check its **date**, **unit of observation** and **measure**. Does its zone-level weekly average answer a question about available connectors at the page's reference time? Is an exact documented API or download offered? State your chosen channel and why.


In [ ]:
import pandas as pd

weekly = pd.read_csv(BASE_URL + "/data/student/weekly_summary.csv")
display(weekly.head())
# TODO: compare the CSV's period and weekly average with the page's reference time and snapshot count.


## Checkpoint 3 — Adapt the prepared Playwright sequence

The browser setup below is provided. Add a wait for **the site list to be ready**, then select **only real, visible site cards**. Inspect one rendered card before writing the extraction loop. Build one record per site with identifier, name, zone, installed and available connectors, status, reference time, collection time and source URL. The final decision uses **available connectors**, not installed capacity or the number of sites marked Operational.


In [ ]:
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=True)
page = await browser.new_page()
await page.goto(STUDENT_URL)
# TODO: wait for the useful page state; avoid a fixed sleep.


In [ ]:
# TODO: identify the section and select only its real visible cards.
card_selector = ""  # Replace with a meaningful locator after inspecting the rendered page.
if not card_selector:
    raise ValueError("Inspect the rendered site list and choose a locator first.")
cards = page.locator(card_selector)
print("Selected cards:", await cards.count())
print("First card:\n", await cards.first.inner_text())


In [ ]:
# TODO: create one dictionary per real site, then make a DataFrame named sites_df.
# Keep the displayed values and their labels distinct: installed != available.


## Checkpoint 4 — Prove the result

Compare your site-row count to the **visible site count**. Compare your numeric totals to the **visible summary**, check site IDs are unique, and spot-check **two individual sites** against the rendered cards. Did your locator include a hidden template or summary box? Record the reference time separately from the collection time.


In [ ]:
# TODO: print counts, unique-ID check, summary comparison and two source spot checks.
# A successful browser run is not yet a validated site-level dataset.


## Checkpoint 5 — Compare zones, document, close

Sum **available connectors by zone** at the page's reference time. Show a two-row table for North and South and recommend the zone with the higher total. Check that the two totals add up to the available-connector count on the page. Keep the recommendation limited to this historical snapshot; the number of available connectors may change, and the nearest site also matters to an actual driver.

Add a short Source Passport (publisher, page, access, reference/collection times, scope, known limits), an AI Audit (suggestion and your verification), and a Defensible Claim (recommendation, evidence, assumptions, limits).


In [ ]:
# TODO: group sites_df by zone and sum available_connectors.
# TODO: display a two-row comparison and recommend a zone using that rule.
# TODO: reconcile the zone totals with the dashboard's available-connector total.
# When finished, release the browser resources:
await browser.close()
await playwright.stop()


**Submission:** completed notebook with a justified source choice, site-level DataFrame, two-row zone comparison, validation evidence and a bounded recommendation for the page's reference time. The annotated solution will be distributed **after** the exercise.
